# Laborator 3 — Exerciții
**Sisteme bazate pe cunoștințe · Calitatea datelor, serii de timp și frecvențe**

Aplicați instrumentele din `Lab3_Exemple_Rezolvate.ipynb` pe date **noi**.

* **Partea A – Detectivul de date** (6 sarcini): `marketplace_listings.csv`, 1 520 de anunțuri cu probleme de calitate ascunse. Încheiați cu un raport de calitate.
* **Partea B – Serii de timp** (5 sarcini): vânzările lunare ale unei farmacii (`dataset.txt`).
* **Partea C – Frecvențe** (4 sarcini): sinusoide ascunse, cererea de electricitate din Los Angeles, scurgerea spectrală, un semnal care își schimbă frecvența.
* **Partea D – Opțional**: date despre bombardamente și vreme în al Doilea Război Mondial (fără autoverificări, doar întrebări).

Fișiere necesare în folderul caietului: `marketplace_listings.csv`, `city_master.csv`, `dataset.txt`, `930-data-export.csv` (iar pentru Partea D `operations.csv`, `Summary of Weather.csv`, `Weather Station Locations.csv`).

Dificultate: ★☆☆ încălzire · ★★☆ standard · ★★★ provocare. Autoverificările afișează ✅ sau ❌ + un indiciu. Numerele se compară **după rotunjirea la 2 zecimale**, iar valorile așteptate sunt stocate ca amprente, deci nu puteți citi răspunsurile – trebuie să le calculați. Rezolvați sarcinile **în ordine**: cele următoare folosesc variabile din cele anterioare. Un traceback roșu la o autoverificare înseamnă doar că sarcina nu e încă rezolvată.

**Predare:** *Kernel → Restart & Run All*, verificați că toate autoverificările sunt ✅ și că **răspunsurile scrise** (celule markdown) și graficele sunt prezente. Predați fișierul `.ipynb`.

In [1]:
%pip install -r ../../requirements.txt


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import hashlib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import signal
from scipy.fft import fft, rfft, rfftfreq

sns.set_theme(style="whitegrid")
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message="The test statistic is outside")   # KPSS: p-valoare la limita tabelului
PINK, BLUE, GREY = "#A33573", "#2A7F9E", "#9AA0A6"

def _norm(v):
    if isinstance(v, (bool, np.bool_)): return str(bool(v))
    if isinstance(v, (int, float, np.integer, np.floating)): return f"{float(v):.2f}"
    return str(v)

def check(label, condition, hint=""):
    """Autoverificare: bifă verde dacă regula e îndeplinită, altfel un indiciu. Nu modificați."""
    print(("✅ " if condition else "❌ ") + label + ("" if condition else f"   → indiciu: {hint}"))

def check_value(label, value, code, hint=""):
    """Compară valoarea voastră (numere rotunjite la 2 zecimale) cu amprenta valorii așteptate."""
    ok = hashlib.sha256(_norm(value).encode()).hexdigest()[:8] == code
    print(("✅ " if ok else "❌ ") + label + ("" if ok else f"   → indiciu: {hint}"))

def make_mystery(seed, fs=200, seconds=4):
    """Partea C: un semnal din trei sinusoide ascunse plus puțin zgomot. Analizați-l, nu citiți codul."""
    rng = np.random.default_rng(seed); t = np.arange(0, seconds, 1 / fs)
    x = 2.5 * np.sin(2 * np.pi * 3 * t) + 1.5 * np.sin(2 * np.pi * 8 * t + 1) + 0.7 * np.sin(2 * np.pi * 21 * t + 2)
    return t, x + 0.05 * rng.normal(size=len(t))

def make_switch(seed, fs=200, seconds=8):
    """Partea C: o sinusoidă care își schimbă frecvența la un moment ascuns."""
    rng = np.random.default_rng(seed); t = np.arange(0, seconds, 1 / fs)
    x = np.where(t < 3.2, np.sin(2 * np.pi * 10 * t), np.sin(2 * np.pi * 40 * t))
    return t, x + 0.05 * rng.normal(size=len(t))

# Partea A — Detectivul de date: anunțuri de piață

## A1 · Prima privire: rânduri, rânduri duplicate, identificatori duplicați  ★☆☆

**Context.** `marketplace_listings.csv` conține anunțurile unei piețe online (aceeași specificație ca în fișa de laborator, plus `Price_EUR` și `Listed_On`). Încărcați-l în `df` (citiți `Rank` ca **text**: `dtype={"Rank": str}`) și răspundeți:

* `n_rows` – numărul de rânduri;
* `n_dup_rows` – numărul de **rânduri complet duplicate** (`duplicated()`; prima apariție *nu* se numără);
* `n_dup_ids` – numărul de rânduri al căror `Unique_Identifier` **a apărut deja mai sus** (`duplicated()` pe coloană);
* `n_rows_sharing_id` – numărul de rânduri al căror identificator apare **de mai multe ori** (`duplicated(keep=False)`).

**Gândiți-vă:** un rând exact duplicat și un *identificator comun cu conținut diferit* sunt probleme diferite. Care este o simplă copie și care ascunde un conflict?

In [63]:
df = pd.read_csv('./marketplace_listings.csv')
n_rows = df.shape[0]
n_dup_rows = df.duplicated().sum()
n_dup_ids = df['Unique_Identifier'].duplicated().sum()
n_rows_sharing_id = df['Unique_Identifier'].duplicated(keep=False).sum()

In [91]:
df.head()

,Listing_ID,Category,Rank,Unique_Identifier,Seller_State,Seller_City,Postal_Code,Price_EUR,Listed_On
0,769,Phone,40.0,P3OKE64E,Karnataka,Magadi,561229,533.05,2026-04-02
1,808,Laptop,7.0,XC6Y1NOI,Karnataka,Magadi,561230,1908.65,2026-03-14
2,1166,Laptop,8.0,P36T1321,Karnataka,Belgaum,590018,859.31,2026-03-17
3,1098,Phone,44.0,DXCUYNVF,Kerala,Kollam,691014,691.94,2026-02-24
4,234,phone,14.0,5D7KL6L2,Karnataka,Hubli,580005,935.82,2026-05-27


In [92]:
n_rows, n_dup_rows, n_dup_ids, n_rows_sharing_id

(1520, np.int64(20), np.int64(35), np.int64(70))

**Autoverificare**

In [65]:
check_value("n_rows", n_rows, "e2f245b4", "len(df)")
check_value("n_dup_rows", n_dup_rows, "b9e68997", "duplicated().sum()")
check_value("n_dup_ids", n_dup_ids, "cbf1326f", "Series.duplicated().sum()")
check_value("n_rows_sharing_id", n_rows_sharing_id, "e60d2ea7", "duplicated(keep=False)")

✅ n_rows
✅ n_dup_rows
✅ n_dup_ids
✅ n_rows_sharing_id


**Interpretarea voastră:** _scrieți aici_

## A2 · Acuratețea coloanelor Category și Rank  ★★☆

Reguli: `Category` ∈ {Phone, Tab, Laptop} (potrivire **exactă**, cu diferență între majuscule și minuscule); `Rank` este un **număr întreg** între 1 și 50.

* `bad_category` – masca booleană a rândurilor care încalcă regula categoriei; `n_bad_category` – câte sunt;
* `rank_num = pd.to_numeric(df["Rank"], errors="coerce")`, apoi
  `n_rank_not_number` (nu s-a putut converti), `n_rank_not_integer` (număr cu parte fracționară), `n_rank_out_of_range` (număr întreg în afara intervalului 1–50).

**Interpretați:** afișați `df.loc[bad_category, "Category"].value_counts()`. Ce fel de greșeli observați (majuscule, sinonime, ortografie)? Le-ar fi găsit pe toate `str.contains("Phone|Tab|Laptop")`?

In [241]:
bad_category = ~df['Category'].isin(['Phone', 'Tab', 'Laptop'])
n_bad_category = bad_category.sum()
rank_num = pd.to_numeric(df['Rank'], errors='coerce')
n_rank_not_number = rank_num.isna().sum()
n_rank_not_integer = (rank_num.notna() & (rank_num % 1 != 0)).sum()
n_rank_out_of_range = (rank_num.notna() & ~rank_num.between(1, 50)).sum()

In [242]:
df.loc[bad_category, "Category"].value_counts()

Category
phone     12
Tablet     8
Mobile     5
Name: count, dtype: int64

In [243]:
n_bad_category, n_rank_not_number, n_rank_not_integer, n_rank_out_of_range

(np.int64(25), np.int64(6), np.int64(8), np.int64(14))

**Autoverificare**

In [244]:
check_value("n_bad_category", n_bad_category, "89b235c0", "isin([...]) cu ~")
check_value("n_rank_not_number", n_rank_not_number, "017144e2", "to_numeric(errors='coerce').isna()")
check_value("n_rank_not_integer", n_rank_not_integer, "ba3109ff", "rang % 1 != 0")
check_value("n_rank_out_of_range", n_rank_out_of_range, "3d03d318", "valoare întreagă și nu between(1, 50)")

✅ n_bad_category
✅ n_rank_not_number
✅ n_rank_not_integer
✅ n_rank_out_of_range


**Interpretarea voastră:** _scrieți aici_

## A3 · Acuratețea coloanelor Unique_Identifier, Seller_State și Postal_Code  ★★☆

Reguli: `Unique_Identifier` = exact 8 litere sau cifre; `Seller_State` ∈ {Karnataka, Kerala}; `Postal_Code` are exact 6 cifre.

* `bad_id` – mască (folosiți `str.fullmatch` cu o expresie regulată); `n_bad_id`;
* `bad_state` – mască; `n_bad_state`;
* `bad_pin_len` – masca codurilor poștale care nu au 6 cifre (`astype(str).str.len()`); `n_bad_pin_len`.

*Atenție:* un identificator lipsă dă `NaN` într-un test pe șiruri; aici nu există, dar gândiți-vă cum le-ați trata.

In [253]:
unique_id = df['Unique_Identifier'].notnull()
bad_id = ~str(unique_id).fullmatch(r'[A-Za-z0-9]{8}')
n_bad_id = ...
bad_state = ...
n_bad_state = ...
bad_pin_len = ...
n_bad_pin_len = ...

AttributeError: 'str' object has no attribute 'fullmatch'

In [254]:
bad_id

0       P3OKE64E
1       XC6Y1NOI
2       P36T1321
3       DXCUYNVF
4       5D7KL6L2
          ...   
1515    E830QMTV
1516    K2J4IOOV
1517    SRWVM1VX
1518    ZP8CPXM4
1519    INDBX89Q
Name: Unique_Identifier, Length: 1520, dtype: str

**Autoverificare**

In [252]:
check_value("n_bad_id", n_bad_id, "df644619", "~str.fullmatch(r'[A-Za-z0-9]{8}')")
check_value("n_bad_state", n_bad_state, "be1019d1", "isin")
check_value("n_bad_pin_len", n_bad_pin_len, "ba3109ff", "astype(str).str.len() != 6")

❌ n_bad_id   → indiciu: ~str.fullmatch(r'[A-Za-z0-9]{8}')
❌ n_bad_state   → indiciu: isin
❌ n_bad_pin_len   → indiciu: astype(str).str.len() != 6


**Interpretarea voastră:** _scrieți aici_

## A4 · Consistență: oraș, stat și cod poștal  ★★★

`city_master.csv` (coloanele `State`, `City`, `PostalCode`) listează combinațiile valide. Lucrați doar cu rândurile care au **stat valid** (`~bad_state`) și `Seller_City` **nelipsă**; numiți acest subset `sub`.

* `n_city_state_mismatch` – rândurile din `sub` a căror pereche (stat, oraș) **nu** se află în lista master;
* `n_prefix_wrong` – rândurile din `sub` cu cod poștal de 6 cifre a căror **prima cifră** nu corespunde statului (Karnataka → 5, Kerala → 6);
* `n_triple_not_in_master` – rândurile din `sub` cu cod de 6 cifre al căror triplet (stat, oraș, cod) nu se află în lista master.

**Indiciu:** construiți un `set` de tupluri din lista master și testați apartenența cu o list comprehension peste `zip(...)`.
**Interpretați:** `n_triple_not_in_master` este mai mare decât `n_city_state_mismatch` sau `n_prefix_wrong` luate separat. De ce găsește testul tripletului mai multe rânduri?

In [ ]:
master = pd.read_csv("city_master.csv")
sub = ...
n_city_state_mismatch = ...
n_prefix_wrong = ...
n_triple_not_in_master = ...

**Autoverificare**

In [ ]:
check_value("n_city_state_mismatch", n_city_state_mismatch, "111ffe69", "perechi (State, City) care nu sunt în lista master")
check_value("n_prefix_wrong", n_prefix_wrong, "a8994a90", "prima cifră față de stat, doar coduri de 6 cifre")
check_value("n_triple_not_in_master", n_triple_not_in_master, "d4ea6b9f", "(State, City, PostalCode) care nu e în lista master, doar coduri de 6 cifre")

**Interpretarea voastră:** _scrieți aici_

## A5 · Completitudine și actualitate  ★★☆

* `n_missing_city`, `n_missing_price` – valori lipsă în `Seller_City` și `Price_EUR`;
* `n_price_nonpositive` – dintre rândurile **cu** preț, câte au prețul **≤ 0**;
* `n_future` – anunțuri cu `Listed_On` **după 30 septembrie 2026** (data exportului). Convertiți cu `pd.to_datetime`.
* `missing_by_category` – un Series cu numărul de rânduri cu preț lipsă pe `Category` (folosiți doar categoriile **valide** `~bad_category`); care categorie e cea mai afectată?

**Interpretați:** un preț 0 este *lipsă*, *greșit* sau *gratuit*? Ce l-ați întreba pe proprietarul datelor?

In [ ]:
n_missing_city = ...
n_missing_price = ...
n_price_nonpositive = ...
listed = pd.to_datetime(df["Listed_On"])
n_future = ...

**Autoverificare**

In [ ]:
check_value("n_missing_city", n_missing_city, "d4ea6b9f", "isna().sum()")
check_value("n_missing_price", n_missing_price, "89b235c0", "isna().sum()")
check_value("n_price_nonpositive", n_price_nonpositive, "c6fffb51", "întâi dropna(), apoi <= 0")
check_value("n_future", n_future, "b2337ecb", "to_datetime(...) > '2026-09-30'")

**Interpretarea voastră:** _scrieți aici_

## A6 · Raportul de calitate  ★★★

Adunați regulile într-un dicționar `rules` (nume → mască booleană peste **toate** rândurile din `df`) și construiți DataFrame-ul `report` cu numărul de încălcări și procentul de rânduri pentru fiecare regulă. Folosiți aceste reguli: categorie greșită, cele trei reguli pentru rang, identificator greșit, stat greșit, nepotrivire oraș–stat, cod poștal fără 6 cifre, prefix poștal greșit, oraș lipsă, preț lipsă, preț ≤ 0, dată din viitor și identificator comun.

Apoi:
* `n_rows_flagged` – rândurile care încalcă **cel puțin una** dintre aceste reguli;
* `share_clean` – ponderea rândurilor (0–1) care nu încalcă **niciuna**.

**Interpretați:** care trei reguli marchează cele mai multe rânduri? Dacă proprietarul poate corecta un singur lucru săptămâna aceasta, ce i-ați recomanda și de ce?

In [ ]:
rules = {}
# completați dicționarul, apoi:
report = ...
n_rows_flagged = ...
share_clean = ...

**Autoverificare**

In [ ]:
check_value("n_rows_flagged", n_rows_flagged, "743973c2", "rândurile cu cel puțin o încălcare: pd.concat(rules, axis=1).any(axis=1)")
check_value("share_clean", share_clean, "4d44dd7e", "1 - marcate / len(df)")

**Interpretarea voastră:** _scrieți aici_

# Partea B — Serii de timp: vânzările lunare ale unei farmacii

## B1 · Încărcarea seriei și alegerea modelului  ★☆☆

`dataset.txt` conține vânzările lunare ale unui medicament (coloanele `date`, `value`), iulie 1991 – iunie 2008.

* `sales` – un `Series` indexat cu **date calendaristice** (`parse_dates`, `index_col`), cu `sales.index.freq = "MS"`;
* `n_months` – lungimea sa; `first_month` – prima dată ca text `"YYYY-MM"` (`strftime`);
* `model` – șirul `"additive"` sau `"multiplicative"`, **decis pe baza dovezilor**: descompuneți cu ambele modele (perioada 12) și comparați mărimea **relativă** a reziduurilor (aditiv: `resid / sales`; multiplicativ: `resid - 1`). Reprezentați mai întâi seria grafic și descrieți ce vedeți.

In [ ]:
sales = ...
n_months = ...
first_month = ...
model = "..."

**Autoverificare**

In [ ]:
check_value("n_months", n_months, "0a8d3dcd", "len(sales)")
check_value("first_month", first_month, "31527002", "sales.index[0].strftime('%Y-%m')")
check_value("model", model, "49959478", "comparați std(resid/sales) la aditiv cu std(resid-1) la multiplicativ")

**Interpretarea voastră:** _scrieți aici_

## B2 · Descompunerea  ★★☆

Descompuneți `sales` cu modelul ales și `period=12`.

* `peak_month` – **numele din trei litere** al lunii cu cel mai mare factor sezonier (de ex. `"Mar"`);
* `peak_factor` – acel factor (2 zecimale);
* `resid_std` – abaterea standard a reziduului (rotunjită cu `round(..., 2)`).

**Interpretați:** de ce diferă atât de mult factorul din ianuarie de cel din februarie? (Indiciu: gândiți-vă la activitatea produsului, de ex. subvenții, începutul anului sau aprovizionarea din decembrie.)

In [ ]:
dec = ...
peak_month = "..."
peak_factor = ...
resid_std = ...

**Autoverificare**

In [ ]:
check_value("peak_month", peak_month, "5c5db120", "idxmax pe primele 12 valori sezoniere, strftime('%b')")
check_value("peak_factor", peak_factor, "f118f9d9", "maximul celor 12 factori sezonieri")
check_value("resid_std", resid_std, "0602d7c8", "dec.resid.std() (NaN-urile se ignoră)")

**Interpretarea voastră:** _scrieți aici_

## B3 · Staționaritatea  ★★☆

Folosiți `adfuller` și `kpss` (din `statsmodels.tsa.stattools`; `kpss(x, regression="c", nlags="auto")`).

* `p_adf_raw` – valoarea p ADF a seriei **brute** (2 zecimale);
* `y` – seria transformată `log(sales)`, prima diferență, **diferența sezonieră de lag 12**, fără NaN;
* `p_adf_y`, `p_kpss_y` – ambele valori p pentru `y` (2 zecimale; KPSS raportează cel mult 0,1);
* `is_stationary` – `True` dacă ADF respinge (p < 0,05) **și** KPSS nu respinge (p ≥ 0,05).

**Interpretați:** scrieți concluzia pentru seria brută și pentru `y`, câte o propoziție, folosind ambele teste.

In [ ]:
from statsmodels.tsa.stattools import adfuller, kpss
p_adf_raw = ...
y = ...
p_adf_y = ...
p_kpss_y = ...
is_stationary = ...

**Autoverificare**

In [ ]:
check_value("p_adf_raw", p_adf_raw, "cf9dcf6d", "adfuller(sales)[1]")
check_value("p_adf_y", p_adf_y, "561b2814", "adfuller(y)[1] pe log + diff + diff(12)")
check_value("p_kpss_y", p_kpss_y, "b4af9cbc", "kpss(..., regression='c', nlags='auto')[1]")
check_value("is_stationary", is_stationary, "3cbc87c7", "ADF p < 0,05 și KPSS p >= 0,05")

**Interpretarea voastră:** _scrieți aici_

## B4 · Autocorelația  ★★★

Cu `y` din B3:

* `acf_vals = acf(y, nlags=24)` (din `statsmodels.tsa.stattools`);
* `lag_max_acf` – lagul (1…24) cu cea mai mare autocorelație în **valoare absolută**;
* `acf_lag12` – autocorelația la lag 12 (2 zecimale);
* `n_sig_lags` – câte dintre lagurile 1…24 sunt semnificative, folosind banda ±1,96/√n (n = len(y)).

Desenați graficele ACF și PACF (`plot_acf`, `plot_pacf`). **Interpretați:** ce înseamnă vârfurile de la lag 1 și lag 12 pentru alegerea unui model (Laboratorul 6)?

In [ ]:
from statsmodels.tsa.stattools import acf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
acf_vals = ...
lag_max_acf = ...
acf_lag12 = ...
n_sig_lags = ...

**Autoverificare**

In [ ]:
check_value("lag_max_acf", lag_max_acf, "cf9dcf6d", "argmax din abs(acf_vals[1:]) + 1")
check_value("acf_lag12", acf_lag12, "e2caa9f3", "acf_vals[12]")
check_value("n_sig_lags", n_sig_lags, "b2337ecb", "abs(acf_vals[1:]) > 1.96 / sqrt(len(y))")

**Interpretarea voastră:** _scrieți aici_

## B5 · Netezirea  ★★☆

Calculați `ma12 = sales.rolling(12, center=True).mean()` și o versiune pe **3 luni**, `ma3`.

* `damping12 = ma12.diff().std() / sales.diff().std()` și `damping3` analog (2 zecimale): cât din mișcarea de la o lună la alta supraviețuiește netezirii?
* `seasonal_left` – abaterea standard a lui `(ma12 / dec.trend) - 1` (2 zecimale; `dec` din B2): dacă este aproape de 0, media pe 12 luni coincide cu trendul din descompunere, adică **sezonalitatea a fost eliminată**.

**Interpretați:** de ce elimină sezonalitatea o fereastră egală cu perioada sezonieră și care este prețul (priviți primele și ultimele luni)?

In [ ]:
ma12 = ...
ma3 = ...
damping12 = ...
damping3 = ...
seasonal_left = ...

**Autoverificare**

In [ ]:
check_value("damping12", damping12, "0602d7c8", "ma12.diff().std() / sales.diff().std()")
check_value("damping3", damping3, "b635f81c", "ma3.diff().std() / sales.diff().std()")
check_value("seasonal_left", seasonal_left, "561b2814", "((ma12 / dec.trend) - 1).std()")

**Interpretarea voastră:** _scrieți aici_

# Partea C — Frecvențe

## C1 · Recuperați trei sinusoide ascunse  ★☆☆

`t, x = make_mystery(1)` returnează 4 secunde dintr-un semnal eșantionat la **200 Hz** (**nu** citiți codul funcției `make_mystery`). Este suma a trei sinusoide plus puțin zgomot.

* `freqs` – cele trei frecvențe în Hz, **sortate**, ca listă de numere întregi rotunjite (folosiți `rfft`, `rfftfreq` și cele mai mari trei vârfuri);
* `amps` – amplitudinile lor, în aceeași ordine, rotunjite la **1 zecimală** (nu uitați să împărțiți la N și să înmulțiți cu 2).

Desenați spectrul unilateral de amplitudine.

In [ ]:
t, x = make_mystery(1)
freqs = ...
amps = ...

**Autoverificare**

In [ ]:
check_value("freqs", freqs, "ac7cb154", "cele mai mari trei vârfuri din abs(rfft(x))/N*2, binurile convertite cu rfftfreq")
check_value("amps", amps, "bc58c461", "abs(rfft(x))/len(x)*2 la acele binuri, 1 zecimală")

**Interpretarea voastră:** _scrieți aici_

## C2 · Cererea de electricitate din Los Angeles  ★★☆

Încărcați `930-data-export.csv` (formatul timpului este în secțiunea 3.4 a fișei: `format="%m/%d/%Y %I %p"` după înlocuirea `a.m.`/`p.m.` și eliminarea ` PDT`). Luați primele **672 de ore** din coloana **`LDWP Demand (MWh)`**, eliminați trendul liniar cu `signal.detrend` și calculați spectrul de amplitudine ca pentru CISO.

* `ldwp_period1`, `ldwp_period2` – perioadele (în ore) ale celor mai mari două vârfuri, ca numere întregi;
* `ldwp_daily_amp` – amplitudinea componentei de 24 h (2 zecimale);
* `ratio_daily` – `ldwp_daily_amp` împărțit la amplitudinea de 24 h a **CISO**, calculată la fel (2 zecimale).

**Interpretați:** LDWP deservește orașul Los Angeles, CISO întregul stat. Ritmul zilnic este *relativ* mai puternic sau mai slab în oraș? Comparați și vârful săptămânal.

In [ ]:
raw = pd.read_csv("930-data-export.csv")
stamp = ...
demand = ...
ldwp_period1 = ...
ldwp_period2 = ...
ldwp_daily_amp = ...
ratio_daily = ...

**Autoverificare**

In [ ]:
check_value("ldwp_period1", ldwp_period1, "85d40e1b", "1 / frecvența celui mai mare vârf, în ore")
check_value("ldwp_period2", ldwp_period2, "c0367b98", "al doilea vârf ca mărime")
check_value("ldwp_daily_amp", ldwp_daily_amp, "150a756e", "amplitudinea la binul cel mai apropiat de 24 h")
check_value("ratio_daily", ratio_daily, "894027e2", "amplitudinea LDWP / amplitudinea CISO la 24 h")

**Interpretarea voastră:** _scrieți aici_

## C3 · Scurgerea spectrală în practică  ★★★

Folosiți `ca = demand["CISO Demand (MWh)"].to_numpy(float)`.

* `amp_672` – amplitudinea celei mai puternice componente cu perioada între 20 și 28 h, folosind primele **672** de ore (fără trend);
* `amp_741` – la fel, folosind **toate cele 741** de ore (fără trend);
* `leak_loss_pct` – cu câte procente este `amp_741` mai mic decât `amp_672` (2 zecimale, `(1 - amp_741/amp_672)*100`).

**Interpretați:** 672 h înseamnă exact 28 de zile, 741 h înseamnă 30,875 zile. Explicați diferența, apoi repetați cu 741 h și o **fereastră Hann** (`signal.windows.hann(n, sym=False)`, împărțiți la media ferestrei, nu la N): repară fereastra amplitudinea?

In [ ]:
ca = demand["CISO Demand (MWh)"].to_numpy(float)
amp_672 = ...
amp_741 = ...
leak_loss_pct = ...

**Autoverificare**

In [ ]:
check_value("amp_672", amp_672, "7e9ac08e", "amplitudinea maximă pentru perioade în [20, 28] h, fără trend, N = 672")
check_value("amp_741", amp_741, "08f12967", "la fel pentru toate cele 741 de ore")
check_value("leak_loss_pct", leak_loss_pct, "ef36ba02", "(1 - amp_741 / amp_672) * 100")

**Interpretarea voastră:** _scrieți aici_

## C4 · Când se schimbă frecvența?  ★★★

`t, x = make_switch(7)` returnează 8 secunde eșantionate la 200 Hz: o sinusoidă care **își schimbă frecvența o singură dată**. Cu FFT simplu vedeți două vârfuri, dar nu *când* are loc schimbarea. Folosiți `signal.stft` (încercați `nperseg = 128` și `256`).

* `f_before`, `f_after` – cele două frecvențe în Hz, **rotunjite la cei mai apropiați 5 Hz** (rezoluția în frecvență a STFT este grosieră);
* `t_switch` – momentul schimbării în secunde (1 zecimală), estimat din spectrogramă, de ex. momentul în care frecvența dominantă trece de mijlocul celor două. Autoverificarea acceptă ±0,3 s.

Desenați spectrograma. **Interpretați:** ce se întâmplă cu claritatea schimbării când dublați `nperseg`?

In [ ]:
t, x = make_switch(7)
f_before = ...
f_after = ...
t_switch = ...

**Autoverificare**

In [ ]:
check_value("f_before", f_before, "a8994a90", "frecvența dominantă din prima parte, în Hz")
check_value("f_after", f_after, "f29c3d3c", "frecvența dominantă din ultima parte, în Hz")
check("t_switch la cel mult 0,3 s de schimbarea reală", abs(t_switch - 3.2) < 0.3, "citiți momentul în care frecvența dominantă sare în spectrogramă")

**Interpretarea voastră:** _scrieți aici_

# Partea D — Opțional: date despre bombardamente și vreme în al Doilea Război Mondial
Date (de pe Kaggle, *Aerial Bombing Operations in World War II* și *Weather Conditions in World War II*): `operations.csv` (178 281 de misiuni, 1939–1945), `Weather Station Locations.csv`, `Summary of Weather.csv` (temperaturi zilnice). Fără autoverificări: răspundeți în celule markdown și justificați cu cod și grafice.

**D1 · Poarta de calitate pentru misiuni.** Încărcați `operations.csv` (`low_memory=False`). Scrieți reguli pentru: `Mission Date` este o dată validă (`pd.to_datetime(errors="coerce")`) între 1939 și 1945; `Takeoff/Target Latitude` în [−90, 90] și `Longitude` în [−180, 180]; `Country`, `Target Country` și `Aircraft Series` nu lipsesc. Construiți un raport de calitate ca la A6. Care coloane sunt în mare parte goale? Ce rânduri ați elimina și pe care le-ați păstra? Justificați.

**D2 · Grafice exploratorii.** Numărul de misiuni pe `Country`; primele 10 `Target Country`; primele 10 `Aircraft Series`; misiuni pe lună (o serie de timp!); locurile de decolare și țintele pe un grafic de împrăștiere longitudine/latitudine.

**D3 · Stație meteo.** Alăturați `Summary of Weather.csv` cu `Weather Station Locations.csv` pe `STA`/`WBAN`. Verificați temperaturile cu reguli de interval (de ex. `MaxTemp ≥ MinTemp`, `MeanTemp` între ele). Alegeți o stație cu o înregistrare lungă, construiți un `Series` **zilnic** cu `MeanTemp` cu index de date și raportați zilele lipsă.

**D4 · Analiza în timp a acelei stații.** Descompuneți (perioada 365 sau, după `resample("MS").mean()`, perioada 12), testați staționaritatea (ADF, KPSS), desenați ACF/PACF și calculați **FFT**: ce perioadă găsiți? Este cea pe care o așteptați pentru temperatură?

**D5 · Legătura dintre cele două.** Reeșantionați misiunile pe luni și comparați numărul lunar de misiuni cu temperatura medie lunară a unei stații din zonă (de ex. Europa). Există o relație? Are sens un test *Granger* aici? De ce (nu)?

## Bonus (opțional) 🎁
1. Rulați funcția de raport de calitate pe fișierul **biroului inteligent** (secțiunea 1.12) și pe coloana `IID` din fișierul cererii. Ce reguli trebuie să adăugați (indiciu: un zero nu este o cerere plauzibilă)?
2. Transformați raportul într-o **funcție** `quality_report(df, rules)` care returnează tabelul și rulați-o din nou după ce curățați datele pieței, ca să demonstrați că problemele au dispărut.

## Reflecție
* Care dintre dimensiunile calității (acuratețe, completitudine, consistență, unicitate, actualitate) a găsit cele mai multe rânduri în Partea A, și care a găsit eroarea *cea mai costisitoare*?
* Ce transformări au fost necesare pentru a face staționară seria farmaciei și ce a eliminat fiecare?
* De ce trebuie eliminate media și trendul înainte de FFT și de ce ar trebui ca înregistrarea să conțină un număr întreg de perioade?